In [7]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

from glob import glob

arc_ex_data = pd.read_csv("valleys.csv").drop(columns=["Unnamed: 0"])
coverage_data = pd.read_csv("coverage-map.csv").drop(columns=["Unnamed: 0"])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
arc_ex_data.head()

,id,fstart,fend,ftype,pers,volume,logvol,majority class,major class size,major class coverage,idx
0,0,-0.0,0.008380,minima-saddle,0.008380,5010,8.519391,cat,5010,0.8350,1
1,2,-0.0,0.004018,minima-saddle,0.004018,5268,8.569596,airplane,5268,0.8780,2
2,3,-0.0,0.004018,minima-saddle,0.004018,5291,8.573952,horse,5291,0.8818,3
3,5,-0.0,0.000972,minima-saddle,0.000972,4569,8.427268,bird,4569,0.7615,4
4,8,-0.0,0.002775,minima-saddle,0.002775,4988,8.514991,deer,4987,0.8312,5


In [51]:
fig = make_subplots(rows=1, cols=1, shared_yaxes=True)

prop = "logvol"
max_v = arc_ex_data[prop].max()
min_v = arc_ex_data[prop].min() - 0.1
vol = (arc_ex_data[prop]) / (max_v)
vol = (arc_ex_data[prop] - min_v) / (max_v - min_v)
colors = list(px.colors.sample_colorscale("OrRd", vol))

for i, row in arc_ex_data.iterrows():
	fig.add_trace(
		go.Scatter(
			x=[row["id"], row["id"]],
			y=[row["fstart"], row["fend"]],
			mode="lines+markers",
			line=dict(color=colors[i], width=2),
			marker=dict(size=10),
		),
		row=1,
		col=1,
	)

fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=14), showlegend=False)
fig.add_vline(x=0.0004965, line_dash="dash", line=dict(color="#C20000"))
fig.update_xaxes(title_text="Valley ID", title_standoff=18, automargin=True)
fig.update_yaxes(title_text="Loss Range", title_standoff=18, automargin=True)
fig.write_image(f"figs/arc-ex.png", scale=8)

In [52]:
coverage_data.head()

,True Label,True Class,count,Proportion,Coverage
0,1,automobile,5642,0.111091,0.9403
1,8,ship,5588,0.110028,0.9313
2,7,horse,5291,0.104180,0.8818
3,0,airplane,5268,0.103727,0.8780
4,6,frog,5032,0.099080,0.8387


In [70]:
new_indices = []

for i, row in coverage_data.iterrows():
    cls = row["True Class"]
    idx = arc_ex_data[arc_ex_data["majority class"] == cls]["idx"].values[0] - 1
    new_indices.append(idx)
    
coverage_data["valley idx"] = new_indices
coverage_data = coverage_data.sort_values(by=["valley idx"])
coverage_data["Coverage"] = coverage_data["Coverage"] * 100
coverage_data.head()

,True Label,True Class,count,Proportion,Coverage,valley idx
5,3,cat,5011,0.098667,83.52,0
3,0,airplane,5268,0.103727,87.80,1
2,7,horse,5291,0.104180,88.18,2
8,2,bird,4569,0.089964,76.15,3
6,4,deer,4987,0.098194,83.12,4


In [88]:
fig = make_subplots(rows=1, cols=1, shared_yaxes=True)

bar = px.bar(coverage_data, x="True Class", y="Coverage", color="True Class", color_discrete_sequence=px.colors.qualitative.Set1)

for trace in bar.data:
	fig.add_trace(trace, row=1, col=1)

fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=250*2, height=200*2, font=dict(size=14), showlegend=False)
fig.add_vline(x=0.0004965, line_dash="dash", line=dict(color="#C20000"))
fig.update_xaxes(title_text="", automargin=True, tickangle=70)
fig.update_yaxes(title_text="Coverage (%)", title_standoff=18, automargin=True, range=[0, 102])
fig.write_image(f"figs/cov.png", scale=8)
# fig.show()